# ビジネス課題の完成版

## Day 1 のプロジェクトを次のレベルへ

### ビジネスチャレンジ：

見込み客・投資家・採用候補者向けに、企業のブローシャーを作成するプロダクトを構築しましょう。

会社名とメインの Web サイト URL を入力として受け取ります。

実際のビジネス応用例はノートブックの末尾をご覧ください。

ご不明な点やアイデアがあれば、いつでもご連絡ください！

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## ステップ 1：関連リンクを GPT-5-nano に判別させる

### GPT-5-nano を呼び出してウェブページのリンクを読み取り、構造化 JSON で返答させます。
関連するリンクを判別し、"/about" のような相対リンクを "https://company.com/about" の形式に変換します。
プロンプト内に回答例を含める「ワンショットプロンプティング」を使用します。

これは LLM の優れたユースケースです。微妙なニュアンスの理解が必要なためです。LLM を使わずにウェブページを解析・分析するコードを書くことを想像してみてください — 非常に困難です！

補足：「Structured Outputs（構造化出力）」というより高度な手法もあります。モデルに仕様に従った回答を強制するものです。この手法は Week 8 の自律型 Agentic AI プロジェクトで扱います。

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## ステップ 2：ブローシャーを作成する！

すべての詳細を別のプロンプトにまとめて GPT-5-nano に渡します

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## 最後に — ちょっとした改善

少し調整するだけで、OpenAI からの結果をストリーミングして、
おなじみのタイプライターアニメーションで表示できるようになります

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">ビジネスへの応用</h2>
            <span style="color:#181;">この演習では Day 1 のコードを拡張して、複数の LLM 呼び出しを行いドキュメントを生成しました。

これはおそらく Agentic AI デザインパターンの最初の例です。複数の LLM 呼び出しを組み合わせたからです。Week 2 でさらに詳しく扱い、Week 8 では完全自律型のエージェントソリューションを構築する際に本格的に取り組みます。

このようなコンテンツ生成は、最もよく使われるユースケースの一つです。要約と同様に、あらゆる業種に応用できます。マーケティングコンテンツの作成、仕様からの製品チュートリアル生成、パーソナライズされたメール文面の作成など、可能性は無限大です。あなたのビジネスにどう応用できるか探り、プルーフ・オブ・コンセプトのプロトタイプを作ってみてください。community-contributions フォルダで他の受講生の作品も確認してみましょう — 価値あるプロジェクトがたくさんあります！</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Week 2 に進む前に（楽しいコンテンツが盛りだくさんです）</h2>
            <span style="color:#900;">Week 1 末尾の EXERCISE ノートブックで Week 1 の課題に挑戦してください。Frontier API を使った実践的な練習ができ、Week 2 への準備が整います。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">3 つの役立つリソースのご案内</h2>
            <span style="color:#f71;">1. コースのリソースは<a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">こちら</a>からご覧いただけます。<br/>
            2. LinkedIn は<a href="https://www.linkedin.com/in/eddonner/">こちら</a>です。コースを受講中の方とつながることを楽しみにしています！<br/>
            3. X/Twitter も始めました。<a href="https://x.com/edwarddonner">@edwarddonner</a> です。使い方を教えてもらえると嬉しいです。
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">最後に！お願いがあります</h2>
            <span style="color:#090;">
                受講生の方が Udemy でこのコースを評価してくださると、コースの露出に大きな違いが生まれます — Udemy が他の人に表示するかどうかを判断する主な基準の一つです。1 分ほどお時間をいただいて評価していただけると、とても助かります！いつでも ed@edwarddonner.com までお気軽にご連絡ください。
            </span>
        </td>
    </tr>
</table>